## Insight from the various information

Many sales person want to know about their counter accounts. And they also want to know the business trend and insights to deliver more relevant solutions to their customers. 

But in many cases, the customers - especially greenfield customers - doesn't want to show their pains and short-term/long-term goals to the sales person in the vendors. 

Our goal is to analyze their business status from their financial statements and pain points from ther crapped news from the internet.



In [1]:
#!pip install google-cloud-aiplatform langchain chroma
#!pip install html5lib
#!pip install beautifulsoup4
#! pip install opendartreader
#!pip install openai

In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

DART_API_KEY=os.getenv("DART_API_KEY")
PROJECT_ID=os.getenv("PROJECT_ID")

In [3]:
import OpenDartReader
from OpenDartReader.dart_list import *

dart = OpenDartReader(DART_API_KEY)
company_list = corp_codes(DART_API_KEY)

In [4]:
import base64
import vertexai
from vertexai.generative_models import GenerativeModel, Part, SafetySetting, FinishReason

finance_analysis_model = GenerativeModel(
  "gemini-1.5-flash-001",
)

generation_config = {
    "max_output_tokens": 8192,
    "temperature": 0.3,
    "top_p": 0.95,
}

safety_settings = [
    SafetySetting(
        category=SafetySetting.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=SafetySetting.HarmBlockThreshold.BLOCK_ONLY_HIGH
    ),
    SafetySetting(
        category=SafetySetting.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=SafetySetting.HarmBlockThreshold.BLOCK_ONLY_HIGH
    ),
    SafetySetting(
        category=SafetySetting.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=SafetySetting.HarmBlockThreshold.BLOCK_ONLY_HIGH
    ),
    SafetySetting(
        category=SafetySetting.HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=SafetySetting.HarmBlockThreshold.BLOCK_ONLY_HIGH
    ),
]


In [19]:
import re
import json

representative_corp_prompt = """당신은 주어진 Data Frame정보에서, 최대한 대표할 만한 회사를 추출해서, corp_code, corp_name 을 반환해 주세요. output format은 json 입니다. 
입력된 값을 임의로 수정하지 말고 출력에 그대로 사용하세요. 
DataFrame을 변경한 값이라, 초기 컬럼이 없는 경우 Index 컬럼이 생략되어 있습니다. 칸을 잘 맞추어서 계산해주세요. 
"""

def get_reprensentative_corp(corp_list):
    response = finance_analysis_model.generate_content(
      [representative_corp_prompt, str(corp_list)],
      generation_config=generation_config,
      safety_settings=safety_settings,
      stream=False,
    )
    text = response.candidates[0].content.text
    return text

def extract_json(text):
    json_pattern = r'```json\s*([\s\S]*?)\s*```'
    match = re.search(json_pattern, text)
    if match:
        json_str = match.group(1)
        return json.loads(json_str)
    return None
  

In [20]:
def searchSimilarStringInDataframeColumns(df_companylist, keyword):
    return df_companylist[df_companylist['corp_name'].str.startswith(keyword)]

In [21]:
TARGET_COMPANY_NAME = 'LX인터내셔널'

In [22]:
similar_company_list=searchSimilarStringInDataframeColumns(company_list, TARGET_COMPANY_NAME)

In [23]:
similar_company_list

,corp_code,corp_name,stock_code,modify_date
87597,00120076,LX인터내셔널,001120,20231115


In [24]:
representative_corp = extract_json(get_reprensentative_corp(str(similar_company_list)))

In [25]:
representative_corp

{'corp_code': '00120076', 'corp_name': 'LX인터내셔널'}

In [26]:
report_list = dart.list(representative_corp['corp_code'], start='1999-01-01', kind='A') 
fs_report_list = report_list[report_list['report_nm'].str.contains('분기보고서') | report_list['report_nm'].str.contains('반기보고서') | report_list['report_nm'].str.contains('사업보고서')]


In [27]:
fs_report_list.head(3)


,corp_code,corp_name,stock_code,corp_cls,report_nm,rcept_no,flr_nm,rcept_dt,rm
0,00120076,LX인터내셔널,001120,Y,반기보고서 (2024.06),20240814003723,LX인터내셔널,20240814,
1,00120076,LX인터내셔널,001120,Y,분기보고서 (2024.03),20240516002080,LX인터내셔널,20240516,
2,00120076,LX인터내셔널,001120,Y,[기재정정]사업보고서 (2023.12),20240320000868,LX인터내셔널,20240320,연


In [28]:
def make_finance_report_df_list(fs_report_list, max_report_num=3):
  reports = []
  for idx, row in fs_report_list.head(max_report_num).iterrows():
    report_html = dart.document(row['rcept_no'])
    report_df = pd.read_html(report_html)
    reports.append((row, report_df))
  return reports

In [29]:
finance_report_df_list = make_finance_report_df_list(fs_report_list, 3)

/tmp/ipykernel_21102/209526689.py:5: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  report_df = pd.read_html(report_html)
/home/postgres/devel/sales_support_bot/.venv/lib/python3.9/site-packages/pandas/io/html.py:661: XMLParsedAsHTMLWarning: It looks like you're parsing an XML document using an HTML parser. If this really is an HTML document (maybe it's XHTML?), you can ignore or filter this warning. If it's XML, you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the lxml package installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.
  soup = BeautifulSoup(udoc, features="html5lib", from_encoding=from_encoding)
/tmp/ipykernel_21102/209526689.py:5: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from

In [30]:
# for meta, one_fs_report in finance_report_df_list:
#   print(meta['report_nm'] + ':' + str(len(str(one_fs_report))))

In [31]:
# def checkNanValueCellCountInDf(df):
#     return df.isnull().sum().sum()

# def checkValidValueCellsInDf(df):
#     return df.count().sum()


In [32]:

prompt_template_agenda = """당신은 재무 분석 전문가입니다. 주어진 재무제표 정보를 바탕으로 해당 기업의 재무적 건전성을 평가해야 합니다. 다음 단계를 따라 분석을 수행하세요:

제공된 재무제표 정보 확인:

대차대조표(재무상태표)
손익계산서(포괄손익계산서)
현금흐름표
주요 재무비율 (제공된 경우)


주요 재무비율 계산 및 분석:

유동비율 = 유동자산 / 유동부채
부채비율 = 총부채 / 자기자본
매출액영업이익률 = 영업이익 / 매출액
총자산이익률(ROA) = 당기순이익 / 총자산
자기자본이익률(ROE) = 당기순이익 / 자기자본


재무상태 평가:

유동성: 단기 부채 상환 능력
레버리지: 부채 수준과 자본 구조
수익성: 이익 창출 능력
효율성: 자산 활용도
성장성: 매출 및 이익 증가율


현금흐름 분석:

영업활동 현금흐름
투자활동 현금흐름
재무활동 현금흐름
전반적인 현금 창출 능력


산업 평균과 비교:

가능한 경우, 주요 재무비율을 산업 평균과 비교


재무적 건전성 종합 평가:

강점과 약점 파악
잠재적 위험 요소 식별
전반적인 재무 건전성에 대한 결론 도출


개선 제안:

재무 상태 개선을 위한 권고사항 제시



이 단계들을 바탕으로 종합적인 분석을 수행하고, 해당 기업의 재무적 건전성에 대한 명확하고 근거 있는 평가를 제공하세요. 출력은 JSON으로 제공해야 합니다.

출력 예시:
{
  "company_name": "회사명",
  "analysis_date": "분석 날짜",
  "financial_ratios": {
    "liquidity_ratio": {
      "current_ratio": 0.0,
      "quick_ratio": 0.0
    },
    "leverage_ratio": {
      "debt_ratio": 0.0,
      "debt_to_equity_ratio": 0.0
    },
    "profitability_ratio": {
      "operating_profit_margin": 0.0,
      "net_profit_margin": 0.0,
      "roa": 0.0,
      "roe": 0.0
    },
    "efficiency_ratio": {
      "asset_turnover": 0.0,
      "inventory_turnover": 0.0
    },
    "growth_ratio": {
      "revenue_growth": 0.0,
      "net_income_growth": 0.0
    }
  },
  "financial_health_assessment": {
    "liquidity": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "유동성 상태에 대한 설명"
    },
    "leverage": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "레버리지 상태에 대한 설명"
    },
    "profitability": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "수익성 상태에 대한 설명"
    },
    "efficiency": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "효율성 상태에 대한 설명"
    },
    "growth": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "성장성 상태에 대한 설명"
    }
  },
  "cash_flow_analysis": {
    "operating_cash_flow": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "영업활동 현금흐름 분석"
    },
    "investing_cash_flow": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "투자활동 현금흐름 분석"
    },
    "financing_cash_flow": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "재무활동 현금흐름 분석"
    },
    "overall_cash_generation": {
      "status": "1~5(매우나쁨~매우좋음)",
      "description": "전반적인 현금 창출 능력 평가"
    }
  },
  "industry_comparison": {
    "description": "산업 평균과의 비교 분석"
  },
  "overall_financial_health": {
    "status": "1~5(매우나쁨~매우좋음)",
    "description": "전반적인 재무 건전성 평가",
    "strengths": [
      "강점 1",
      "강점 2"
    ],
    "weaknesses": [
      "약점 1",
      "약점 2"
    ],
    "potential_risks": [
      "잠재적 위험 1",
      "잠재적 위험 2"
    ]
  },
  "recommendations": [
    "개선 제안 1",
    "개선 제안 2"
  ]
}

<finance statement>
"""


In [33]:
prompt_template_agenda_post = """
</finance statement>

<instruction>
1. 모든 항목을 자세하게 읽고 답변해 주세요. 
2. 답변에 대해서 다시 한번 검토를 하고 이상이 없는지 확인 후 답변을 다시 제출해 주세요.
3. 모든 비율은 정확하게 구성하여 주세요.
4. 모든 비율은 소수점 둘째자리까지 표기해 주세요.
5. 모든 비율은 스트링으로 표기해 주세요.
6. 두개 이상 분기의 재무재표 정보가 들어갈 수 있으므로, 이에 대해서 정확하게 구분해서 처리해 주세요.
7. status 는 1점에 5정 사이의 값으로 표기해 주세요.
"""

In [34]:


def make_finance_analysis(finance_report_df_list):
  report_text = ""
  for meta, one_fs_report in finance_report_df_list:
    one_report_text = """
<{finance_report_nm}>
{one_fs_report}
</{finance_report_nm}>
""".format(finance_report_nm=meta['report_nm'], one_fs_report=str(one_fs_report))
    report_text += one_report_text

  response = finance_analysis_model.generate_content(
      [prompt_template_agenda, report_text, prompt_template_agenda_post],
      generation_config=generation_config,
      safety_settings=safety_settings,
      stream=False,
  )
  text = response.candidates[0].content.text
  print(text)
  return text


In [35]:
finance_health_analysis = extract_json(make_finance_analysis(finance_report_df_list))

```json
{
  "company_name": "㈜LX인터내셔널",
  "analysis_date": "2024-08-14",
  "financial_ratios": {
    "liquidity_ratio": {
      "current_ratio": "1.40",
      "quick_ratio": "1.38"
    },
    "leverage_ratio": {
      "debt_ratio": "0.62",
      "debt_to_equity_ratio": "1.64"
    },
    "profitability_ratio": {
      "operating_profit_margin": "0.31",
      "net_profit_margin": "0.27",
      "roa": "0.24",
      "roe": "0.62"
    },
    "efficiency_ratio": {
      "asset_turnover": "0.88",
      "inventory_turnover": "16.40"
    },
    "growth_ratio": {
      "revenue_growth": "-22.63",
      "net_income_growth": "-75.19"
    }
  },
  "financial_health_assessment": {
    "liquidity": {
      "status": "4",
      "description": "유동비율과 당좌비율 모두 1 이상으로 단기 부채 상환 능력이 양호함. 특히, 당좌비율은 1.38로 단기 부채 상환 능력이 매우 안정적임."
    },
    "leverage": {
      "status": "3",
      "description": "부채비율은 0.62로 산업 평균 대비 다소 높은 수준이나, 자기자본 대비 부채 비율은 1.64로 적정 수준임. 과도한 부채 의존도는 아니지만, 부채 규모를 관리하는 데 주의가 필요함."
    },
    "

In [37]:
prompt_business_trend_analysis = """
당신은 기술 및 비즈니스 전략 분석가입니다. 주어진 기업 정보를 바탕으로 해당 기업의 기술/비즈니스 환경 변화에 대한 대응 능력과 전략을 평가해야 합니다. 다음 단계를 따라 분석을 수행하고, 결과를 JSON 형식으로 출력하세요:

제공된 기업 정보 확인:

기업 개요 및 주요 제품/서비스
최근 기술 투자 및 R&D 활동
시장 점유율 및 경쟁사 정보
최근 비즈니스 모델 변화 또는 전략적 결정


기술 혁신 및 적응력 평가:

신기술 도입 및 활용 현황
R&D 투자 효율성
특허 및 지적 재산권 포트폴리오
기술 인재 확보 및 유지 전략


시장 대응력 분석:

시장 트렌드 대응 속도
고객 니즈 변화에 대한 대응 능력
새로운 시장 진출 전략
경쟁사 대비 차별화 전략


디지털 전환 수준 평가:

디지털 기술 활용도
데이터 기반 의사결정 체계
온라인/모바일 플랫폼 전략
AI, 빅데이터 등 첨단 기술 활용 현황


비즈니스 모델 혁신:

새로운 수익 모델 개발 능력
기존 비즈니스 모델의 지속가능성
산업 간 경계를 넘는 혁신 시도
협업 및 파트너십 전략


환경/사회적 대응:

지속가능성 전략 및 ESG 대응
환경 규제 대응 능력
사회적 책임 활동 및 평판 관리


리스크 관리 및 회복탄력성:

기술적/시장적 리스크 대응 체계
글로벌 불확실성에 대한 대비책
위기 상황 대응 능력 및 사례


JSON 형식으로 결과 출력:
아래 제시된 JSON 구조를 따라 분석 결과를 출력하세요. 각 섹션에 대한 상세한 설명과 1(매우 나쁨)에서 5(매우 좋음)까지의 평가 점수를 포함하세요.

output example :
{
  "company_name": "회사명",
  "analysis_date": "분석 날짜",
  "technological_innovation": {
    "status": 0,
    "description": "기술 혁신 및 적응력에 대한 평가",
    "key_strengths": ["강점1", "강점2"],
    "areas_for_improvement": ["개선점1", "개선점2"]
  },
  "market_responsiveness": {
    "status": 0,
    "description": "시장 대응력에 대한 분석",
    "key_strategies": ["전략1", "전략2"],
    "challenges": ["도전과제1", "도전과제2"]
  },
  "digital_transformation": {
    "status": 0,
    "description": "디지털 전환 수준 평가",
    "successful_initiatives": ["성공사례1", "성공사례2"],
    "improvement_areas": ["개선필요영역1", "개선필요영역2"]
  },
  "business_model_innovation": {
    "status": 0,
    "description": "비즈니스 모델 혁신 평가",
    "innovative_approaches": ["혁신적 접근1", "혁신적 접근2"],
    "potential_risks": ["잠재적 위험1", "잠재적 위험2"]
  },
  "environmental_social_response": {
    "status": 0,
    "description": "환경/사회적 대응 평가",
    "esg_initiatives": ["ESG 이니셔티브1", "ESG 이니셔티브2"],
    "areas_of_concern": ["우려사항1", "우려사항2"]
  },
  "risk_management_resilience": {
    "status": 0,
    "description": "리스크 관리 및 회복탄력성 평가",
    "effective_measures": ["효과적인 조치1", "효과적인 조치2"],
    "vulnerabilities": ["취약점1", "취약점2"]
  },
  "overall_assessment": {
    "status": 0,
    "description": "전반적인 기술/비즈니스 환경 대응 능력 평가",
    "key_competitive_advantages": ["핵심 경쟁 우위1", "핵심 경쟁 우위2"],
    "critical_challenges": ["중요 과제1", "중요 과제2"]
  },
  "recommendations": [
    "권고사항1",
    "권고사항2",
    "권고사항3"
  ]
}

<finance statement>
"""

In [38]:
prompt_business_trend_analysis_post = """
</finance statement>

<instruction>
1. 모든 항목을 자세하게 읽고 답변해 주세요. 
2. 답변에 대해서 다시 한번 검토를 하고 이상이 없는지 확인 후 답변을 다시 제출해 주세요.
3. 모든 비율은 정확하게 구성하여 주세요.
4. 모든 비율은 소수점 둘째자리까지 표기해 주세요.
5. 모든 비율은 스트링으로 표기해 주세요.
6. 두개 이상 분기의 재무재표 정보가 들어갈 수 있으므로, 이에 대해서 정확하게 구분해서 처리해 주세요.
7. status 는 1점에 5정 사이의 값으로 표기해 주세요.
"""

In [39]:
def make_biz_tech_analysis(finance_report_df_list):
  report_text = ""
  for meta, one_fs_report in finance_report_df_list:
    one_report_text = """
<{finance_report_nm}>
{one_fs_report}
</{finance_report_nm}>
""".format(finance_report_nm=meta['report_nm'], one_fs_report=str(one_fs_report))
    report_text += one_report_text

  response = finance_analysis_model.generate_content(
      [prompt_business_trend_analysis, report_text, prompt_business_trend_analysis_post],
      generation_config=generation_config,
      safety_settings=safety_settings,
      stream=False,
  )
  text = response.candidates[0].content.text
  print(text)
  return text


In [40]:
biz_trend_analysis = extract_json(make_biz_tech_analysis(finance_report_df_list))

```json
{
  "company_name": "㈜LX인터내셔널",
  "analysis_date": "2024-08-16",
  "technological_innovation": {
    "status": 3,
    "description": "㈜LX인터내셔널은 디지털 전환을 위한 기술 투자와 R&D 활동을 적극적으로 추진하고 있지만, 첨단 기술 활용은 아직 초기 단계이며, 기술 인재 확보 및 유지 전략은 미흡한 편입니다.",
    "key_strengths": [
      "인도네시아 PT. Energy Battery Indonesia (PT. EBI) 출자를 통한 배터리 사업 진출 시도",
      "인도네시아 PT. Adhi Kartiko Pratama (PT. AKP) 지분 60% 취득을 통한 니켈 광산 사업 확장"
    ],
    "areas_for_improvement": [
      "AI, 빅데이터 등 첨단 기술 활용 확대 및 전문 인력 확보",
      "기술 기반의 새로운 사업 모델 개발 및 시장 진출 전략 수립"
    ]
  },
  "market_responsiveness": {
    "status": 4,
    "description": "㈜LX인터내셔널은 시장 트렌드에 대한 대응 속도가 빠르며, 고객 니즈 변화에 대한 대응 능력도 뛰어납니다. 특히, 신성장 사업 발굴을 통한 새로운 시장 진출 전략을 적극적으로 추진하고 있습니다.",
    "key_strategies": [
      "인도네시아 에듀테크 시장 진출을 위한 온라인 교육 플랫폼 개발",
      "인도네시아 진단검사 시장 진출을 위한 현지 JV법인 설립 및 운영",
      "친환경 사업 추진을 위한 친환경 고체연료(SRF) 제조시설 및 폐기물 선별시설 설립"
    ],
    "challenges": [
      "신성장 사업의 수익성 확보 및 지속 가능성 확보",
      "경쟁 심화에 대한 대응 전략 강화"
    ]
  },
 

In [41]:
prompt_cost_analysis = """
당신은 기업의 원가 분석 담당자입니다. 주어진 재무 및 운영 데이터를 바탕으로 상세한 원가 분석과 수익성 평가를 수행해야 합니다. 다음 단계를 따라 분석을 수행하고, 결과를 JSON 형식으로 출력하세요:

제공된 데이터 확인:

제품/서비스별 매출 정보 (판매량, 판매단가)
원가 구조 (직접재료비, 직접노무비, 제조간접비)
판매관리비 내역
기타 비용 항목


제품/서비스별 원가 분석:

단위당 직접재료비, 직접노무비, 제조간접비 계산
제품/서비스별 총원가 및 단위원가 산출
원가 구조 분석 (각 원가 요소의 비중)


판매량 및 판매단가 분석:

제품/서비스별 판매량 추이 분석
판매단가의 변동 추이 및 영향 요인 분석
판매믹스 분석 (전체 매출에서 각 제품/서비스의 비중)


수익성 분석:

제품/서비스별 매출총이익 및 이익률 계산
손익분기점 분석
공헌이익 분석
영업이익 및 순이익 분석


원가 동인 분석:

주요 원가 동인 식별
원가 동인과 원가 간의 관계 분석
원가 절감 기회 도출


효율성 분석:

생산성 지표 분석 (예: 노동생산성, 설비생산성)
재고회전율 분석
자산회전율 분석


경쟁사 및 산업 평균과의 비교:

주요 원가 및 수익성 지표의 경쟁사 비교
산업 평균과의 격차 분석


JSON 형식으로 결과 출력:
아래 제시된 JSON 구조를 따라 분석 결과를 출력하세요. 각 섹션에 대한 상세한 설명과 1(매우 나쁨)에서 5(매우 좋음)까지의 평가 점수를 포함하세요.

output example :
{
  "company_name": "회사명",
  "analysis_date": "분석 날짜",
  "product_service_analysis": [
    {
      "name": "제품/서비스명",
      "unit_cost": {
        "direct_material": 0.0,
        "direct_labor": 0.0,
        "manufacturing_overhead": 0.0,
        "total": 0.0
      },
      "selling_price": 0.0,
      "sales_volume": 0,
      "gross_profit": 0.0,
      "gross_profit_margin": 0.0,
      "status": 0,
      "description": "제품/서비스별 원가 및 수익성 분석"
    }
  ],
  "cost_structure_analysis": {
    "direct_material_ratio": 0.0,
    "direct_labor_ratio": 0.0,
    "manufacturing_overhead_ratio": 0.0,
    "sga_expense_ratio": 0.0,
    "status": 0,
    "description": "전반적인 원가 구조 분석"
  },
  "sales_analysis": {
    "total_revenue": 0.0,
    "revenue_growth_rate": 0.0,
    "sales_mix": [
      {
        "product_service": "제품/서비스명",
        "revenue_share": 0.0
      }
    ],
    "status": 0,
    "description": "판매량 및 판매단가 분석"
  },
  "profitability_analysis": {
    "gross_profit": 0.0,
    "gross_profit_margin": 0.0,
    "operating_profit": 0.0,
    "operating_profit_margin": 0.0,
    "net_profit": 0.0,
    "net_profit_margin": 0.0,
    "break_even_point": 0.0,
    "contribution_margin_ratio": 0.0,
    "status": 0,
    "description": "전반적인 수익성 분석"
  },
  "cost_driver_analysis": {
    "key_cost_drivers": ["원가 동인1", "원가 동인2"],
    "cost_reduction_opportunities": ["기회1", "기회2"],
    "status": 0,
    "description": "주요 원가 동인 및 절감 기회 분석"
  },
  "efficiency_analysis": {
    "labor_productivity": 0.0,
    "asset_turnover": 0.0,
    "inventory_turnover": 0.0,
    "status": 0,
    "description": "생산성 및 효율성 분석"
  },
  "competitive_analysis": {
    "cost_position": {
      "status": 0,
      "description": "경쟁사 대비 원가 포지션"
    },
    "profitability_position": {
      "status": 0,
      "description": "경쟁사 대비 수익성 포지션"
    }
  },
  "overall_assessment": {
    "status": 0,
    "description": "전반적인 원가 및 수익성 평가",
    "key_strengths": ["강점1", "강점2"],
    "areas_for_improvement": ["개선점1", "개선점2"]
  },
  "recommendations": [
    "권고사항1",
    "권고사항2",
    "권고사항3"
  ]
}


<finance statement>
"""

In [42]:
prompt_cost_analysis_post = """
</finance statement>

<instruction>
1. 모든 항목을 자세하게 읽고 답변해 주세요. 
2. 답변에 대해서 다시 한번 검토를 하고 이상이 없는지 확인 후 답변을 다시 제출해 주세요.
3. 모든 비율은 정확하게 구성하여 주세요.
4. 모든 비율은 소수점 둘째자리까지 표기해 주세요.
5. 모든 비율은 스트링으로 표기해 주세요.
6. 두개 이상 분기의 재무재표 정보가 들어갈 수 있으므로, 이에 대해서 정확하게 구분해서 처리해 주세요.
7. status 는 1점에 5정 사이의 값으로 표기해 주세요.
"""

In [43]:
def make_cost_analysis(finance_report_df_list):
  report_text = ""
  for meta, one_fs_report in finance_report_df_list:
    one_report_text = """
<{finance_report_nm}>
{one_fs_report}
</{finance_report_nm}>
""".format(finance_report_nm=meta['report_nm'], one_fs_report=str(one_fs_report))
    report_text += one_report_text

  response = finance_analysis_model.generate_content(
      [prompt_cost_analysis, report_text, prompt_cost_analysis_post],
      generation_config=generation_config,
      safety_settings=safety_settings,
      stream=False,
  )
  text = response.candidates[0].content.text
  print(text)
  return text

In [44]:
cost_analysis = extract_json(make_cost_analysis(finance_report_df_list))

```json
{
  "company_name": "㈜LX인터내셔널",
  "analysis_date": "2024-08-14",
  "product_service_analysis": [
    {
      "name": "광물사업",
      "unit_cost": {
        "direct_material": 0.0,
        "direct_labor": 0.0,
        "manufacturing_overhead": 0.0,
        "total": 0.0
      },
      "selling_price": 0.0,
      "sales_volume": 0,
      "gross_profit": 0.0,
      "gross_profit_margin": "0.00",
      "status": 3,
      "description": "제품/서비스별 원가 및 수익성 분석 데이터 부족"
    },
    {
      "name": "팜사업",
      "unit_cost": {
        "direct_material": 0.0,
        "direct_labor": 0.0,
        "manufacturing_overhead": 0.0,
        "total": 0.0
      },
      "selling_price": 0.0,
      "sales_volume": 0,
      "gross_profit": 0.0,
      "gross_profit_margin": "0.00",
      "status": 3,
      "description": "제품/서비스별 원가 및 수익성 분석 데이터 부족"
    },
    {
      "name": "자원 Trading",
      "unit_cost": {
        "direct_material": 0.0,
        "direct_labor": 0.0,
        "manufacturing_overhead": 0.

In [49]:
prompt_investment_consultant = """당신은 기업 전략 컨설턴트입니다. 재무, 기술/비즈니스 환경, 원가 분석 리포트를 종합하여 해당 기업의 전반적인 평가를 수행해야 합니다. 다음 단계를 따라 분석을 수행하고, 결과를 JSON 형식으로 출력하세요:

제공된 리포트 검토:

재무 건전성 분석 리포트
기술/비즈니스 환경 변화 분석 리포트
원가 분석 및 수익성 평가 리포트


재무적 성과 종합:

수익성, 유동성, 레버리지 등 주요 재무 지표 평가
재무적 강점과 약점 식별


기술 및 시장 대응력 평가:

기술 혁신 능력 및 디지털 전환 수준 평가
시장 변화에 대한 대응 능력 분석


원가 구조 및 수익성 분석:

원가 경쟁력 평가
수익성 개선 가능성 분석


전략적 포지션 평가:

산업 내 경쟁 우위 요소 식별
장기적 성장 잠재력 평가


리스크 요인 분석:

재무적, 기술적, 시장적 리스크 요인 식별
리스크 관리 능력 평가


종합 SWOT 분석:

강점(Strengths), 약점(Weaknesses), 기회(Opportunities), 위협(Threats) 도출


JSON 형식으로 결과 출력:
아래 제시된 JSON 구조를 따라 분석 결과를 출력하세요. 각 섹션에 대한 상세한 설명과 1(매우 나쁨)에서 5(매우 좋음)까지의 평가 점수를 포함하세요.


{
  "company_name": "회사명",
  "analysis_date": "분석 날짜",
  "financial_performance": {
    "status": 0,
    "description": "전반적인 재무 성과 평가",
    "key_strengths": ["강점1", "강점2"],
    "key_weaknesses": ["약점1", "약점2"]
  },
  "technology_market_responsiveness": {
    "status": 0,
    "description": "기술 및 시장 대응력 평가",
    "innovation_capacity": {
      "status": 0,
      "description": "기술 혁신 능력 평가"
    },
    "market_adaptability": {
      "status": 0,
      "description": "시장 변화 적응력 평가"
    }
  },
  "cost_structure_profitability": {
    "status": 0,
    "description": "원가 구조 및 수익성 평가",
    "cost_competitiveness": {
      "status": 0,
      "description": "원가 경쟁력 평가"
    },
    "profitability_improvement_potential": {
      "status": 0,
      "description": "수익성 개선 잠재력"
    }
  },
  "strategic_position": {
    "status": 0,
    "description": "산업 내 전략적 포지션 평가",
    "competitive_advantages": ["경쟁 우위1", "경쟁 우위2"],
    "growth_potential": {
      "status": 0,
      "description": "장기적 성장 잠재력 평가"
    }
  },
  "risk_assessment": {
    "status": 0,
    "description": "전반적인 리스크 평가",
    "key_risk_factors": ["리스크 요인1", "리스크 요인2"],
    "risk_management_capacity": {
      "status": 0,
      "description": "리스크 관리 능력 평가"
    }
  },
  "swot_analysis": {
    "strengths": ["강점1", "강점2"],
    "weaknesses": ["약점1", "약점2"],
    "opportunities": ["기회1", "기회2"],
    "threats": ["위협1", "위협2"]
  },
  "overall_assessment": {
    "status": 0,
    "description": "기업의 종합적인 평가",
    "key_findings": ["주요 발견사항1", "주요 발견사항2"],
    "future_outlook": "기업의 미래 전망에 대한 서술"
  },
  "recommendations": [
    {
      "area": "개선 영역",
      "recommendation": "구체적인 권고사항",
      "priority": "고/중/저"
    }
  ]
}


"""

In [48]:
prompt_investment_consultant_post = """

이 구조화된 JSON 형식으로 분석 결과를 출력하세요. 각 섹션에 대해 상세하고 정확한 정보를 제공하되, 제공된 세 가지 리포트의 내용을 종합적으로 고려하여 판단하세요. 모든 'status' 필드는 1(매우 나쁨)에서 5(매우 좋음)까지의 정수로 평가해야 합니다.
분석 시 다음 사항을 고려하세요:

각 분야의 평가가 서로 어떻게 연관되는지 파악하세요.
단기적 성과와 장기적 잠재력을 균형있게 평가하세요.
산업 특성과 시장 동향을 고려하여 평가하세요.
정량적 데이터와 정성적 분석을 적절히 조합하세요.
객관적이고 공정한 시각을 유지하되, 중요한 인사이트를 제공하세요.

최종적으로, 기업의 현재 상황에 대한 명확한 그림을 제시하고, 미래 성장을 위한 실행 가능한 권고사항을 제공하세요.

"""

In [47]:
def generate_overall_company_analysis(finance_health_analysis, biz_trend_analysis, cost_analysis):
  response = finance_analysis_model.generate_content(
      [prompt_investment_consultant, """<finance_health_analysis>
{finance_health_analysis}
</finance_health_analysis>

      """
      , """<biz_trend_analysis>

{biz_trend_analysis}
</biz_trend_analysis>

      """,
      """<cost_analysis>
{cost_analysis}
</cost_analysis>
  
        """
      , prompt_investment_consultant_post],
      generation_config=generation_config,
      safety_settings=safety_settings,
      stream=False,
  )
  text = response.candidates[0].content.text
  print(text)
  return text
 


In [50]:
overall_review = extract_json(generate_overall_company_analysis(finance_health_analysis, biz_trend_analysis, cost_analysis))

```json
{
  "company_name": "회사명",
  "analysis_date": "2023-10-26",
  "financial_performance": {
    "status": 3,
    "description": "재무 건전성 분석 결과, 회사는 안정적인 수익성을 유지하고 있지만 유동성과 레버리지 측면에서 개선 여지가 있습니다.",
    "key_strengths": [
      "안정적인 매출 성장세",
      "높은 이익률"
    ],
    "key_weaknesses": [
      "낮은 유동비율",
      "높은 부채 비율"
    ]
  },
  "technology_market_responsiveness": {
    "status": 2,
    "description": "기술 혁신 능력은 다소 부족하며, 시장 변화에 대한 대응 속도가 느린 편입니다.",
    "innovation_capacity": {
      "status": 2,
      "description": "R&D 투자는 적극적이지만, 시장 트렌드를 선도하는 혁신적인 기술 개발은 부족합니다."
    },
    "market_adaptability": {
      "status": 2,
      "description": "시장 변화에 대한 민감도가 낮고, 신규 시장 진출이나 사업 모델 변화에 대한 적응력이 부족합니다."
    }
  },
  "cost_structure_profitability": {
    "status": 4,
    "description": "원가 경쟁력은 높은 편이며, 수익성 개선을 위한 잠재력이 존재합니다.",
    "cost_competitiveness": {
      "status": 4,
      "description": "원자재 조달 및 생산 효율성이 높아 원가 경쟁력이 우수합니다."
    },
    "profitability_improvement_potential": {
    